# 5 · Demo and interpretability

Runs the browser demo from Colab and inspects what drives each prediction.

**Person D deliverable.**


## Setup

Clone the repository and install. The data layer needs nothing beyond the standard
library, so this is only for the model code.


In [ ]:
!git clone -q https://github.com/ManasDasri/NNDL.git
%cd NNDL
!pip install -q -e . 'matplotlib>=3.8'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > Change runtime type > T4 GPU')


### Prepare the corpus

Extracts answer spans from `CUAD_v1.json` and assigns document-level splits.
Chunking happens at training time, because each model needs a different window.


In [ ]:
!legal-risk-prepare --cuad_json data/CUAD_v1.json --out_dir data/processed


In [ ]:
!pip install -q 'gradio>=4.44'


### Launch the demo

`--share` prints a public link that works for about 72 hours. Paste a contract,
get the six labels scored, the flagged passages quoted, and the phrases that
triggered each flag.


In [ ]:
!legal-risk-demo --run_dir outputs/cnn --share


### Why the model said what it said

A convolutional filter is a literal n-gram detector and max-over-time pooling
records which position fired it hardest, so an explanation here is an actual phrase
from the contract rather than a diffuse weight over tokens.


In [ ]:
from legal_risk_classifier.runtime import CNNRuntime
from legal_risk_classifier.attribution import explain
from legal_risk_classifier.cuad import load_documents
from legal_risk_classifier.splits import load_splits, select

runtime = CNNRuntime('outputs/cnn')
documents = select(load_documents('data/processed/documents.jsonl'),
                   load_splits('data/processed/splits.json'), 'test')

doc = next(d for d in documents if d.spans['Insurance'])
prediction = runtime.predict_document(doc.text, doc.doc_id)
print(doc.doc_id)
print('annotated:', doc.labels())
print('predicted:', prediction.predicted)

for label in ('Insurance', 'Cap on Liability'):
    hits = prediction.hits(label, threshold=runtime.threshold_for(label))
    print(f'\n--- {label} ---')
    for phrase in explain(runtime.model, runtime.vocab,
                          hits[0].text if hits else doc.text, label, top_n=5):
        print(f'  {phrase.contribution:6.2f}  {phrase.text!r}')


### A defect this exposes

Ask why it flagged *Cap on Liability* and it returns the same `product liability
insurance` phrases it uses for *Insurance*. It is keying on the shared word
"liability" rather than on capping language such as *"in no event shall"* or
*"shall not exceed"*.

That is a concrete lead on the per-class results rather than a guess about them.


### Rebuild the dashboard

Regenerates the results page from whatever runs now exist.


In [ ]:
!legal-risk-export-dashboard --run_dir outputs/cnn
